<a href="https://colab.research.google.com/github/UdayLab/cuPAMI/blob/main/notebooks/frequentPattern/clApriori.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding frequent pattern using clApriori

This tutorial shows how to prepare a dataset, run clApriori, inspect the generated patterns, and save the results.

[Algorithm documentation](https://github.com/udayRage/cuPAMI/blob/main/docs/algorithms/clApriori.md)

# Prerequisites

1. Installing cuPAMI and PyOpenCL

Use a machine with a compatible OpenCL GPU runtime. On AMD, install the vendor GPU driver and OpenCL runtime first. Colab uses NVIDIA GPUs, so running this notebook there validates only the NVIDIA OpenCL path. CUDA and CuPy are not dependencies of clApriori. AMD performance has not been validated.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("cupami") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/UdayLab/cuPAMI.git"])
if importlib.util.find_spec("pyopencl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyopencl"])


2. Checking the GPU

In [2]:
from cupami import clApriori

# Only GPU devices are listed. This does not select an OpenCL CPU backend.
devices = clApriori.listDevices()
for device in devices:
    print(device)
if not devices:
    raise RuntimeError("Install a compatible vendor OpenCL GPU runtime before continuing.")


{'index': 0, 'name': 'NVIDIA RTX A6000', 'vendor': 'NVIDIA Corporation', 'platform': 'NVIDIA CUDA', 'version': 'OpenCL 3.0 CUDA'}
{'index': 1, 'name': 'NVIDIA RTX A6000', 'vendor': 'NVIDIA Corporation', 'platform': 'NVIDIA CUDA', 'version': 'OpenCL 3.0 CUDA'}


3. Preparing a sample dataset

Use BinaryTransactionalDatabase with a fixed seed so the input can be reproduced.

In [3]:
from cupami import BinaryTransactionalDatabase, clApriori

generator = BinaryTransactionalDatabase(transactions=100, items=8, avg_length=3, seed=42)
data = generator.create()
print(data[:5])


[[2, 5, 6, 7], [1, 2, 4, 5], [1], [2, 4, 7], [1, 2, 7]]


## Finding patterns using clApriori

### Step 1: Choosing the input parameters

`minSup=0.1` means 10% of nonempty transactions. Set `vendor="AMD"` to require an AMD OpenCL GPU. With `vendor=None`, AMD is preferred, otherwise another OpenCL GPU is used; the selected device is printed below.

### Step 2: Executing clApriori

In [4]:
vendor = None  # Set to "AMD" on your AMD GPU machine.
miner = clApriori(data, minSup=0.1, vendor=vendor)
miner.mine()
print(miner.getComputeBackend())


{'backend': 'opencl', 'device': 'NVIDIA RTX A6000', 'vendor': 'NVIDIA Corporation', 'platform': 'NVIDIA CUDA', 'work_group_size': 32}


### Step 3: Investigating the generated patterns

Display the number of patterns and the first ten results.

In [5]:
from itertools import islice

patterns = miner.getPatterns()
print("Patterns:", len(patterns))
for pattern, support in islice(patterns.items(), 10):
    print(pattern, support)


Patterns: 35
1 44
2 45
3 34
4 29
5 39
6 48
7 36
8 37
1	2 22
1	3 18


### Step 4: Saving patterns and measuring runtime

Save patterns to `patterns.txt`. Times are in seconds. The first mining call includes OpenCL context creation and kernel compilation; warmed calls reuse them. The generated data and file output are outside the mining timer.

In [6]:
print("Loading:", miner.getLoadingTime())
print("Mining:", miner.getMiningTime())
print("Total:", miner.getTotalRuntime())
miner.save("patterns.txt")


Loading: 9.2889997176826e-05
Mining: 0.14628386800177395
Total: 0.14637675799895078
